<a id="feature_selection"></a>
# Features Selection - Hugo Fiévet

Le but de cette partie est de sélectionner les variables que le modèle va utiliser par la suite.

Je vais présenter 4 méthodes de sélection des variables à savoir :

- Sélection personnelle (**P**)
- Sélection via Variance Threshold (**VT**)
- Sélection via Select K-Best (**KB**)

Dans un premier temps j'affiche les variables disponibles dans mon train set.

In [ ]:
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

X_train = pd.read_csv("../data/processed/xtrain.csv")
X_test = pd.read_csv("../data/processed/xtest.csv")

X_train.columns.to_list()

print(f'Numbers of variables : {len(X_train.columns.to_list())}')



['koi_period',
 'koi_period_err1',
 'koi_period_err2',
 'koi_time0bk',
 'koi_time0bk_err1',
 'koi_time0bk_err2',
 'koi_impact',
 'koi_impact_err1',
 'koi_impact_err2',
 'koi_duration',
 'koi_duration_err1',
 'koi_duration_err2',
 'koi_depth',
 'koi_depth_err1',
 'koi_depth_err2',
 'koi_prad',
 'koi_prad_err1',
 'koi_prad_err2',
 'koi_teq',
 'koi_insol',
 'koi_insol_err1',
 'koi_insol_err2',
 'koi_model_snr',
 'koi_steff',
 'koi_steff_err1',
 'koi_steff_err2',
 'koi_slogg',
 'koi_slogg_err1',
 'koi_slogg_err2',
 'koi_srad',
 'koi_srad_err1',
 'koi_srad_err2',
 'dec',
 'koi_kepmag',
 'ra_sin',
 'ra_cos',
 'koi_tce_delivname_q1_q16_tce',
 'koi_tce_delivname_q1_q17_dr24_tce',
 'koi_tce_delivname_q1_q17_dr25_tce',
 'koi_tce_plnt_num_1.0',
 'koi_tce_plnt_num_2.0',
 'koi_tce_plnt_num_3.0',
 'koi_tce_plnt_num_4.0',
 'koi_tce_plnt_num_5.0',
 'koi_tce_plnt_num_6.0',
 'koi_tce_plnt_num_7.0',
 'koi_tce_plnt_num_8.0',
 'koi_fpflag_nt',
 'koi_fpflag_ss',
 'koi_fpflag_co',
 'koi_fpflag_ec']

Numbers of variables : 51


## <u>Personal selection (**P**):</u>





Je commence par ma sélection personnelle, en m'appuyant sur l'analyse exploratoire des données réalisée mais aussi sur mon intuition.

| Variable / famille de variables| Sélectionnée (Oui / Non) | Justification |
|----------|--------------------------|---------------|
|`koi_fpflag_* `| Non : pour tous | Car si le modèle apprend à partir des flags, il ne découvre pas réellement les caractéristiques physiques qui permettent de distinguer les classes. Il exploite une information déjà très proche de la décision finale, autrement dit de la target `koi_disposition`.|
|`*_err_1 et *_err_2`|Non : pour tous|Je souhaite privilégier les caractéristiques physiques principales des candidats et des étoiles hôtes. Bien que les incertitudes apportent une information sur la précision des mesures, je choisis de ne pas les intégrer à ma sélection personnelle afin de limiter le nombre de variables et la redondance potentielle avec les mesures auxquelles elles sont associées.|
|`koi_time0bk`|Non|Cette variable indique l'époque d'un transit de référence. Cette information permet de positionner temporellement les transits, mais ne décrit pas directement les propriétés physiques permettant de distinguer un candidat planétaire d'un faux positif. Je choisis donc de l'exclure de ma sélection personnelle.|
|Mesures du candidat : `koi_period`, `koi_impact`, `koi_duration`, `koi_depth`|Oui : pour tous|Ces variables décrivent des caractéristiques complémentaires du transit. Même si leurs distributions présentent individuellement des chevauchements entre les classes, leur combinaison peut aider le modèle à mieux distinguer les différents types de signaux.|
|Qualité du signal de transit : `koi_model_snr`|Oui|`koi_model_snr` est conservée car le rapport signal/bruit renseigne sur la qualité de détection du transit. L’EDA montre des différences de distribution entre les classes, même si elles restent insuffisantes pour les distinguer seules. Cette variable peut donc apporter une information discriminante lorsqu’elle est combinée aux autres caractéristiques du transit.|
|`koi_teq`,`koi_insol`|`koi_teq` : Non ; `koi_insol` : Oui | `koi_teq` et `koi_insol` sont très fortement liées statistiquement et physiquement. Pour limiter la redondance dans ma sélection personnelle, je ne conserve qu'une des deux. Je choisis koi_insol, qui représente directement le flux énergétique reçu par la planète, tandis que koi_teq est une température d'équilibre estimée à partir de ce bilan énergétique et d'hypothèses physiques.|
|`koi_prad`|Oui|L'EDA montre que koi_prad apporte une information discriminante entre les différentes classes, malgré un chevauchement important de leurs distributions. Elle est donc conservée comme variable potentiellement pertinente pour la classification.|
|`koi_steff`|Oui|Malgré un fort chevauchement entre les classes, `koi_steff` décrit une propriété physique propre à l’étoile hôte et peut apporter une information complémentaire aux caractéristiques du transit et du candidat. Elle est donc conservée dans la sélection personnelle.|
|`koi_slogg`|Non|L'EDA montre des distributions presque identiques, ce qui n'aiderait pas à faciliter la détection d'un signal.|
|`koi_srad`|Oui|La dispersion est différente et concentrée dans une certaine plage pour la modalité `CONFIRMED`.|
|`koi_kepmag`|Oui|Contrairement à `koi_steff` ou `koi_slogg`, elle concerne directement les conditions dans lesquelles Kepler mesure le signal lumineux. C’est complémentaire des caractéristiques du transit.|
|`ra` et `dec`|Non|`ra` et `dec` décrivent uniquement la position de l’étoile dans le ciel et ne constituent pas des caractéristiques physiques du transit, du candidat ou de l’étoile hôte directement liées à sa disposition. Je choisis donc de les exclure de ma sélection personnelle afin d’éviter que le modèle apprenne d’éventuels effets spatiaux ou instrumentaux propres au champ d’observation de Kepler plutôt que les caractéristiques du signal.|
|`koi_tce_delivname_*`|Non|Je supprime `koi_tce_delivname`, car cette variable décrit la version du catalogue/processus de détection dont provient le TCE et non une propriété physique du candidat. Elle pourrait introduire un biais lié au processus de traitement des données.|
|`koi_tce_plnt_num_*`|Non|Je supprime `koi_tce_plnt_num`, car cette variable correspond principalement à l'organisation des TCE associés à une même cible plutôt qu'à une caractéristique physique du signal. Je privilégie dans ma sélection personnelle les informations décrivant directement le transit, le candidat ou son étoile hôte.|

Ma sélection personnelle sera composée de **10** variables.


In [13]:
LEAKAGE_FEATURES = [
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec"
]

X_train_clean = X_train.drop(columns=LEAKAGE_FEATURES)
X_test_clean = X_test.drop(columns=LEAKAGE_FEATURES)

P_SELECTED_COLUMNS = ["koi_period", "koi_impact", "koi_duration", "koi_depth",
                    "koi_model_snr","koi_insol","koi_prad",
                    "koi_steff","koi_kepmag","koi_srad"
                    ]

X_train_P = X_train[P_SELECTED_COLUMNS]
X_test_P = X_test[P_SELECTED_COLUMNS]

## <u>Variance Threshold selection (**VT**):</u>





In [14]:
from sklearn.feature_selection import VarianceThreshold

vt_selector = VarianceThreshold(threshold=0.01)

X_train_VT_array = vt_selector.fit_transform(X_train_clean)

X_test_VT_array = vt_selector.transform(X_test_clean)

selected_features_VT = X_train_clean.columns[
    vt_selector.get_support()
]

X_train_VT = pd.DataFrame(
    X_train_VT_array,
    columns=selected_features_VT,
    index=X_train.index
)

X_test_VT = pd.DataFrame(
    X_test_VT_array,
    columns=selected_features_VT,
    index=X_test.index
)

removed_features_VT = X_train_clean.columns[
    ~vt_selector.get_support()
]

print("Variables removed:")
print(removed_features_VT.tolist())

print("\nTrain shape:", X_train.shape, "->", X_train_VT.shape)
print("Test shape :", X_test.shape, "->", X_test_VT.shape)



Variables removed:
['koi_tce_plnt_num_5.0', 'koi_tce_plnt_num_6.0', 'koi_tce_plnt_num_7.0', 'koi_tce_plnt_num_8.0']

Train shape: (7651, 51) -> (7651, 43)
Test shape : (1913, 51) -> (1913, 43)


## <u>Select K-Best selection (**KB**):</u>





In [ ]:
from sklearn.feature_selection import SelectKBest,f_classif



y_train = pd.read_csv("../data/processed/ytrain.csv").squeeze("columns")

f_scores, p_values = f_classif(X_train_clean, y_train)

kbest_scores = pd.DataFrame({
    "Feature": X_train_clean.columns,
    "F_score": f_scores,
    "p_value": p_values
}).sort_values("F_score", ascending=False)

kbest_scores

kb_selector = SelectKBest(
    score_func=f_classif,
    k=15
)

X_train_KB_array = kb_selector.fit_transform(
    X_train_clean,
    y_train
)

X_test_KB_array = kb_selector.transform(
    X_test_clean
)

selected_features_KB = X_train_clean.columns[
    kb_selector.get_support()
]

X_train_KB = pd.DataFrame(
    X_train_KB_array,
    columns=selected_features_KB,
    index=X_train_clean.index
)

X_test_KB = pd.DataFrame(
    X_test_KB_array,
    columns=selected_features_KB,
    index=X_test_clean.index
)

print(selected_features_KB.tolist())
print(X_train_clean.shape, "->", X_train_KB.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'ytrain.csv'

Les variables ont été classées selon leur F-score obtenu avec **f_classif**. J'ai choisi de conserver les **15** variables ayant les scores les plus élevés afin de retenir les caractéristiques présentant les différences les plus marquées entre les classes, tout en réduisant sensiblement la dimension du jeu de données. Le choix reste volontairement conservateur afin de ne pas supprimer trop d'information.

# CSV export

In [ ]:
def save_selected_data(dataframes):
    """
    Enregistre les jeux d'entraînement et de test issus
    des différentes méthodes de sélection de variables.

    Parameters
    ----------
    dataframes : dict
        Dictionnaire associant le nom du fichier au DataFrame à enregistrer.
    """
    for file_name, X_set in dataframes.items():
        X_set.reset_index(drop=True).to_csv(
            f"../data/selected/{file_name}.csv",
            index=False
        )
        
selected_datasets = {
    "xtrain_P": X_train_P,
    "xtest_P": X_test_P,
    "xtrain_VT": X_train_VT,
    "xtest_VT": X_test_VT,
    "xtrain_KB": X_train_KB,
    "xtest_KB": X_test_KB
}

save_selected_data(selected_datasets)

print("Files created:")
print("==============")
print("- xtrain_P.csv")
print("- xtest_P.csv")

print("- xtrain_VT.csv")
print("- xtest_VT.csv")

print("- xtrain_KB.csv")
print("- xtest_KB.csv")



Files created:
- xtrain_P.csv
- xtest_P.csv
- xtrain_VT.csv
- xtest_VT.csv
- xtrain_KB.csv
- xtest_KB.csv
